# creditlib on Google Colab

Single-name CDS pricing engine: hazard-rate bootstrapping, exact leg integration, credit risk measures.

**Run the cells top to bottom.** Setup takes about a minute.

> **Colab runtimes are ephemeral.** Files vanish when the runtime disconnects (~90 min idle). You'll
> re-run Step 1 each session. The permanent fix is Option B below — put the repo on GitHub and clone it,
> which also gives you a link you can put on a résumé.

## Step 1 — Get the code

Pick **one** of the two options.

### Option A — Upload the zip (fastest, do this first time)

Run the cell, click **Choose Files**, select `creditlib.zip`.

In [ ]:
import os, sys, zipfile, shutil

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("Not running in Colab -- skip to Option B or use a local checkout.")

if IN_COLAB:
    if os.path.exists('/content/creditlib'):
        shutil.rmtree('/content/creditlib')
    uploaded = files.upload()                      # select creditlib.zip
    name = next(iter(uploaded))
    with zipfile.ZipFile(name) as z:
        z.extractall('/content')
    print("\nExtracted to /content/creditlib")

### Option B — Clone from GitHub (do this once you've pushed the repo)

Replace `YOUR-USERNAME` and run. This is the version to use long term: no re-uploading every session,
and you get a shareable Colab link (see the last cell).

In [ ]:
# !rm -rf /content/creditlib
# !git clone -q https://github.com/YOUR-USERNAME/creditlib.git /content/creditlib
# print("cloned")

## Step 2 — Install

Colab already has numpy, scipy, pandas and matplotlib. Only QuantLib is missing, and it is a
**validation-only** dependency — never on the pricing path. If it fails to install, everything still
works; the parity tests just skip.

In [ ]:
%pip install -q QuantLib pytest

import sys
sys.path.insert(0, '/content/creditlib')     # simpler than `pip install -e .` in Colab

import creditlib
print("creditlib loaded from:", creditlib.__file__)

try:
    import QuantLib as ql
    print("QuantLib", ql.__version__, "-- validation tests will run")
except ImportError:
    print("QuantLib absent -- 8 parity tests will skip, everything else runs")

## Step 3 — Verify

Expect `86 passed`, or `78 passed, 8 skipped` if QuantLib didn't install. Anything else means
something is wrong — don't trust the numbers below until this is green.

In [ ]:
!cd /content/creditlib && python -m pytest tests/ -q 2>&1 | tail -5

## Step 4 — See it work

The quickstart walks the four workflows: build a discount curve, price a CDS, bootstrap from market
quotes, compute risk. It ends with sanity anchors you can verify on paper.

In [ ]:
!cd /content/creditlib && python quickstart.py

## Step 5 — Use it yourself

Everything below is live. Change the quotes, the recovery, the tenors.

In [ ]:
from datetime import date
import numpy as np
from creditlib import (DiscountCurve, bootstrap_survival_curve, make_cds_schedule,
                       price_cds, par_spread, cs01, jump_to_default, spread_bounds)

# --- your inputs -----------------------------------------------------------
QUOTES    = {1.0: 0.0055, 3.0: 0.0082, 5.0: 0.0105, 7.0: 0.0118, 10.0: 0.0130}
RECOVERY  = 0.40
TRADE     = date(2026, 8, 17)
NOTIONAL  = 10_000_000
COUPON    = 0.0100          # standard IG fixed coupon
# ---------------------------------------------------------------------------

discount = DiscountCurve.from_zero_rates(
    [0.25, 0.5, 1, 2, 3, 5, 7, 10],
    [0.0430, 0.0421, 0.0402, 0.0381, 0.0374, 0.0372, 0.0378, 0.0390],
)

boot = bootstrap_survival_curve(QUOTES, discount, RECOVERY, trade_date=TRADE)
print(boot)

In [ ]:
sched = make_cds_schedule(TRADE, tenor_years=5.0)
print(price_cds(discount, boot.curve, COUPON, RECOVERY, sched, notional=NOTIONAL))

print(f"\nCS01 {cs01(discount, boot.curve, COUPON, RECOVERY, sched, NOTIONAL):>14,.2f}")
print(f"JTD  {jump_to_default(discount, boot.curve, COUPON, RECOVERY, sched, NOTIONAL):>14,.2f}")

### Plot the curve

In [ ]:
import matplotlib.pyplot as plt

t = np.linspace(0, 10, 400)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4.2))

a1.plot(t, (1 - np.asarray(boot.curve.survival(t))) * 100, lw=2, color="#1F4E79")
a1.plot(boot.knot_times, (1 - boot.survival) * 100, "o", color="#1F4E79", ms=7,
        mec="white", mew=1.3, zorder=5, label="quoted tenors")
a1.set_xlabel("Horizon (years)"); a1.set_ylabel("Cumulative default probability (%)")
a1.set_title("Bootstrapped PD curve", loc="left"); a1.grid(alpha=.25); a1.legend(frameon=False)

a2.step(np.concatenate(([0], boot.knot_times)),
        np.concatenate((boot.hazards[:1], boot.hazards)) * 1e4,
        where="pre", lw=2, color="#9A4B00")
a2.set_xlabel("Horizon (years)"); a2.set_ylabel("Forward hazard rate (bp)")
a2.set_title("Piecewise-flat hazards -- FORWARD, not average", loc="left"); a2.grid(alpha=.25)

plt.tight_layout(); plt.show()

### Check your quotes are arbitrage-free before bootstrapping

Each quote is bounded above and below by the shorter quotes. Outside the window, no non-negative
hazard reproduces it — that's an arbitrage in the quote set, not a solver problem.

In [ ]:
for k, tenor in enumerate(QUOTES):
    lo, hi = spread_bounds(QUOTES, discount, RECOVERY, tenor_index=k, trade_date=TRADE)
    ok = "ok" if lo < QUOTES[tenor] < hi else "OUT OF BOUNDS"
    print(f"{tenor:5.1f}Y  [{lo*1e4:8.2f}, {hi*1e4:10.2f}] bp   quoted {QUOTES[tenor]*1e4:7.2f}   {ok}")

### What a bad quote set looks like

In [ ]:
from creditlib import InvertedCurveError, SpreadCeilingError

bad = dict(QUOTES); bad[3.0] = 0.0010          # 3Y far below the 1Y
try:
    bootstrap_survival_curve(bad, discount, RECOVERY, trade_date=TRADE)
except InvertedCurveError as e:
    print("InvertedCurveError:\n ", e)

bad = dict(QUOTES); bad[3.0] = 4.0             # 3Y at 40,000bp
try:
    bootstrap_survival_curve(bad, discount, RECOVERY, trade_date=TRADE)
except SpreadCeilingError as e:
    print("\nSpreadCeilingError:\n ", e)

## Step 6 — The applied analysis

Opens the five-bank notebook. In Colab: **File → Open notebook → Upload**, then pick
`notebooks/bank_credit_analysis.ipynb` from the zip. Or, once the repo is on GitHub, use the
GitHub tab in the same dialog.

---

## Making this permanent

Colab wipes `/content` on disconnect. Two things worth doing:

**1. Push to GitHub.** Then Option B replaces the upload step, and you get a link of the form

```
https://colab.research.google.com/github/YOUR-USERNAME/creditlib/blob/main/notebooks/creditlib_colab.ipynb
```

which anyone can open and run without installing anything. That link is a far better portfolio
artifact than a zip attachment.

```bash
# run locally, once
cd creditlib
git init
printf '.venv/\n__pycache__/\n*.egg-info/\n.pytest_cache/\n.ipynb_checkpoints/\n' > .gitignore
git add -A && git commit -m "creditlib: single-name CDS pricing engine"
git branch -M main
git remote add origin https://github.com/YOUR-USERNAME/creditlib.git
git push -u origin main
```

**2. Save a copy of this notebook** to your Drive: **File → Save a copy in Drive**. Otherwise your
edits here vanish too.

## If something breaks

| Symptom | Fix |
|---|---|
| `ModuleNotFoundError: creditlib` | Re-run Step 1 — the runtime probably disconnected and wiped `/content` |
| QuantLib install fails | Ignore it. 71 tests still pass; parity tests skip |
| Tests fail after you edit code | Check the sanity anchors at the end of `quickstart.py` — the credit triangle should converge to 120bp and repricing error should be ~1e-16 |
| Plots don't render | Re-run the cell; Colab occasionally drops the first figure |